In [10]:
# Import relevant libraries
import os
import numpy as np
import pandas as pd
import dabest
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

# Import summary_ci_1group for individual group mean calculations
from dabest._stats_tools.confint_1group import summary_ci_1group

## FUNCTION

In [11]:
# File paths
officecomp = "C:\\Users\\Star\\"
labcomp = "C:\\Users\\User\\"
computer2 = "C:\\Users\\lnico\\"
homecomp = "D:\\"
specifiedpath = homecomp

# Input directory (compiled OSAR files)
input_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\2025collection\\"

# Output directory for thesis stats
output_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\thesis_stats\\"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# List all files
files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
print(f"Found {len(files)} files")
print(files[:5])  # Show first 5

Found 60 files
['MB018B x ACR.csv', 'MB018B x Chrimson2.csv', 'MB077B x ACR.csv', 'MB077B x Chrimson2.csv', 'MB082C x ACR.csv']


In [12]:
## Thesis OSAR Function - Creates standardized DataFrames for thesis

def thesis_osar_hedgesg(df, metric_col, df_naming, driver, responder):
    """
    Process OSAR data using hedges_g for each light intensity.
    Returns 8 rows: 4 light intensities x 2 groups (Control, Test)
    """
    try:
        light_intensities = ["Eighth", "Quarter", "Half", "Full"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        
        rows = []
        
        for intensity in light_intensities:
            # Filter data for this light intensity
            df_intensity = df[df['light_intensity'] == intensity].copy()
            
            # Skip if no data for this intensity
            if len(df_intensity) == 0:
                continue
            
            # Skip if metric column doesn't exist
            if metric_col not in df_intensity.columns:
                continue
            
            # Remove NaN and infinite values for the metric BEFORE passing to dabest
            df_intensity = df_intensity[np.isfinite(df_intensity[metric_col])]
            
            if len(df_intensity) == 0:
                continue
            
            # Separate Sibling (Control) vs Offspring (Test)
            df_control = df_intensity[df_intensity['status'] == 'Sibling']
            df_test = df_intensity[df_intensity['status'] == 'Offspring']
            
            # Skip if either group is empty
            if len(df_control) == 0 or len(df_test) == 0:
                continue
            
            # Prepare data for dabest
            df_intensity['Group'] = df_intensity['status'].map({'Sibling': 'Control', 'Offspring': 'Test'})
            
            # Load dabest for hedges_g comparison
            db = dabest.load(df_intensity, idx=("Control", "Test"), y=metric_col, x='Group')
            results = db.hedges_g.results
            
            # Check if results are valid
            if len(results) == 0:
                continue
            
            # Get plot data for mean calculations
            plot_data = db._plot_data
            xvar = db._xvar
            yvar = db._yvar
            
            # Process Control and Test groups
            groups = ["Control", "Test"]
            genotypes = [genotype_control, genotype_test]
            
            for group, genotype in zip(groups, genotypes):
                # Get group data for mean calculation - ensure it's a proper numpy float array
                group_data = plot_data[plot_data[xvar] == group][yvar].values
                group_data = np.array(group_data, dtype=float)
                
                if len(group_data) == 0:
                    continue
                
                # Filter out any remaining NaN/inf (should already be clean, but just in case)
                group_data = group_data[np.isfinite(group_data)]
                if len(group_data) == 0:
                    continue
                
                # Calculate mean and CI
                group_stats = summary_ci_1group(
                    x=group_data,
                    func=np.mean,
                    resamples=5000,
                    alpha=0.05,
                    random_seed=12345
                )
                
                mean_str = f"{group_stats['summary']:.2f}\n[{group_stats['bca_ci_low']:.2f}, {group_stats['bca_ci_high']:.2f}]"
                sample_size = len(group_data)
                
                # Effect Size - only for Test row (use .iloc for proper indexing)
                if group == "Test":
                    effect_size_str = f"{results.difference.iloc[0]:.2f}\n[{results.bca_low.iloc[0]:.2f}, {results.bca_high.iloc[0]:.2f}]"
                    delta_object = "Hedges' g"
                else:
                    effect_size_str = " "
                    delta_object = " "
                
                rows.append({
                    "MBON": driver,
                    "Responder": responder,
                    "Light Intensity": intensity,
                    "Group": group,
                    "Genotype": genotype,
                    "Sample Size": sample_size,
                    "Mean": mean_str,
                    "Effect Size": effect_size_str,
                    "Delta Object": delta_object,
                    "Delta-Delta/Delta-g": " ",  # Always blank for OSAR (no delta2)
                    "Metric": df_naming
                })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_osar_hedgesg for {driver} - {df_naming}: {e}")
        return pd.DataFrame()

In [4]:
# Process all OSAR files

# Metrics to process (pace renamed to bout)
metrics = [
    ("pi_smoothed_Pattern 01", "pi"),
    ("light_attraction_index_Pattern 01", "light_attraction_index"),
    ("speed_ratio_Pattern 01", "speed_ratio"),
    ("log2_speed_ratio_Pattern 01", "log2_speed_ratio"),
    ("pace_ratio_Pattern 01", "bout_speed_ratio"),  # pace = bout speed
    ("log2_pace_ratio_Pattern 01", "log2_bout_speed_ratio"),  # pace = bout speed
    ("bout_index_Pattern 01", "bout_number_index"),
    ("bout_duration_ratio_Pattern 01", "bout_duration_ratio"),
    ("max_velocity_ratio_Pattern 01", "max_velocity_ratio"),
]

# Store all results
all_thesis_dfs = []

for file in files:
    # Parse filename to get driver and responder
    # Format: "MB112C x ACR.csv" -> driver="MB112C", responder="ACR"
    filename_no_ext = file.replace('.csv', '')
    parts = filename_no_ext.split(' x ')
    
    if len(parts) != 2:
        print(f"Skipping file with unexpected format: {file}")
        continue
    
    driver = parts[0].strip()
    responder = parts[1].strip()
    
    print(f"Processing: {driver} x {responder}")
    
    # Read the CSV file
    df = pd.read_csv(input_dir + file)
    
    # Process each metric
    driver_dfs = []
    for metric_col, df_naming in metrics:
        try:
            metric_df = thesis_osar_hedgesg(df, metric_col, df_naming, driver, responder)
            if len(metric_df) > 0:
                driver_dfs.append(metric_df)
        except Exception as e:
            print(f"  Error processing {df_naming} for {driver}: {e}")
    
    # Concatenate all metrics for this driver
    if driver_dfs:
        driver_thesis_df = pd.concat(driver_dfs, ignore_index=True)
        all_thesis_dfs.append(driver_thesis_df)
        
        # Save individual driver file
        driver_thesis_df.to_csv(output_dir + f"{driver} x {responder}_thesis_stats.csv", index=False)

print("\nDone processing all files!")

Processing: MB018B x ACR
Processing: MB018B x Chrimson2
Processing: MB077B x ACR
Processing: MB077B x Chrimson2
Processing: MB082C x ACR
Processing: MB082C x Chrimson2
Processing: MB093C x ACR
Processing: MB093C x Chrimson2
Processing: MB112C x ACR
Processing: MB112C x Chrimson2
Processing: MB210B x Chrimson2
Processing: MB242A x ACR
Processing: MB242A x Chrimson2
Processing: MB310C x ACR
Processing: MB310C x Chrimson2
Processing: MB319C x ACR
Processing: MB319C x Chrimson2
Processing: MB323B x ACR
Processing: MB323B x Chrimson2
Processing: MB434B x ACR
Processing: MB434B x Chrimson2
Processing: R76B09 x ACR
Processing: R76B09 x Chrimson2
Processing: SS01188 x ACR
Processing: SS01188 x Chrimson2
Processing: SS01194 x Chrimson2
Processing: SS01298 x ACR
Processing: SS01298 x Chrimson2
Processing: SS01308 x ACR
Processing: SS01308 x Chrimson2
Processing: SS46348 x ACR
Processing: SS46348 x Chrimson2
Processing: SS52050 x ACR
Processing: SS52050 x Chrimson2
Processing: SS67662 x ACR
Proce

## Total file

In [17]:
df_osar_thesis_all = pd.DataFrame()
individualfiles = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\thesis_stats\\"
indi_files = os.listdir(individualfiles)
for j in indi_files:
    all_thesis_dfs = pd.read_csv(individualfiles + j)
    df_osar_thesis_all = pd.concat([df_osar_thesis_all, all_thesis_dfs], ignore_index=True)
    
df_osar_thesis_all.to_csv(output_dir + f"{date}_all_osar_thesis_stats.csv", index=False)

## total file - if ran individual files


In [7]:
# Combine all drivers into one master DataFrame
if all_thesis_dfs:
    df_osar_thesis_all = pd.concat(all_thesis_dfs, ignore_index=True)
    
    # Save master file
    df_osar_thesis_all.to_csv(output_dir + f"{date}_all_osar_thesis_stats.csv", index=False)
    
    print(f"Total rows: {len(df_osar_thesis_all)}")
    print(f"Unique drivers: {df_osar_thesis_all['MBON'].nunique()}")
    print(f"Saved to: {output_dir}{date}_all_osar_thesis_stats.csv")
    
    df_osar_thesis_all.head(20)

Total rows: 4320
Unique drivers: 32
Saved to: D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\Data Compilation\osar_compiled\thesis_stats\20260119_all_osar_thesis_stats.csv


In [18]:
df_osar_thesis_all

,MBON,Responder,Light Intensity,Group,Genotype,Sample Size,Mean,Effect Size,Delta Object,Delta-Delta/Delta-g,Metric
0,MB018B,ACR,Eighth,Control,w1118;;UAS-ACR/+\nw1118;MB018B/+;MB018B/+,89,"-0.05\n[-0.05, -0.05]",,,,pi
1,MB018B,ACR,Eighth,Test,w1118;MB018B/+;MB018B/ACR,60,"0.15\n[0.14, 0.14]","0.28\n[-0.08, 0.64]",Hedges' g,,pi
2,MB018B,ACR,Quarter,Control,w1118;;UAS-ACR/+\nw1118;MB018B/+;MB018B/+,102,"-0.11\n[-0.11, -0.11]",,,,pi
3,MB018B,ACR,Quarter,Test,w1118;MB018B/+;MB018B/ACR,60,"0.20\n[0.20, 0.20]","0.49\n[0.12, 0.88]",Hedges' g,,pi
4,MB018B,ACR,Half,Control,w1118;;UAS-ACR/+\nw1118;MB018B/+;MB018B/+,101,"-0.19\n[-0.19, -0.19]",,,,pi
...,...,...,...,...,...,...,...,...,...,...,...
4315,VT999036,Chrimson2,Quarter,Test,w1118;VT999036/+;VT999036/Chrimson2,17,"1.19\n[1.20, 1.20]","0.59\n[-0.39, 1.62]",Hedges' g,,max_velocity_ratio
4316,VT999036,Chrimson2,Half,Control,w1118;;UAS-Chrimson2/+\nw1118;VT999036/+;VT999...,103,"1.12\n[1.12, 1.12]",,,,max_velocity_ratio
4317,VT999036,Chrimson2,Half,Test,w1118;VT999036/+;VT999036/Chrimson2,9,"0.95\n[0.95, 0.95]","-0.66\n[-1.60, 0.39]",Hedges' g,,max_velocity_ratio
4318,VT999036,Chrimson2,Full,Control,w1118;;UAS-Chrimson2/+\nw1118;VT999036/+;VT999...,106,"1.05\n[1.05, 1.05]",,,,max_velocity_ratio


In [6]:
# View sample output for one driver
if all_thesis_dfs:
    print(df_osar_thesis_all[df_osar_thesis_all['MBON'] == 'MB112C'].to_string())

       MBON  Responder Light Intensity    Group                                         Genotype  Sample Size                   Mean            Effect Size Delta Object Delta-Delta/Delta-g                  Metric
576  MB112C        ACR          Eighth  Control        w1118;;UAS-ACR/+\nw1118;MB112C/+;MB112C/+           70     0.07\n[0.07, 0.07]                                                                              pi
577  MB112C        ACR          Eighth     Test                        w1118;MB112C/+;MB112C/ACR           30  -0.02\n[-0.02, -0.02]   -0.13\n[-0.56, 0.31]    Hedges' g                                          pi
578  MB112C        ACR         Quarter  Control        w1118;;UAS-ACR/+\nw1118;MB112C/+;MB112C/+           70  -0.06\n[-0.06, -0.06]                                                                              pi
579  MB112C        ACR         Quarter     Test                        w1118;MB112C/+;MB112C/ACR           30  -0.32\n[-0.31, -0.31]   -0.40\n[-0.81